# Walk-Forward Out-of-Sample Evaluation and Long-Short Backtest

## Objective

Evaluate the frozen OLS, Ridge, Elastic Net, and XGBoost specifications
on the untouched 2022+ test period.

The evaluation uses an expanding annual walk-forward procedure:

- At the beginning of each year, fit each model using all observations
  whose targets would already have been known.
- Keep all previously selected hyperparameters fixed.
- Generate predictions for the coming year.
- Add the completed year to the training history before the next refit.

The final evaluation considers both:

1. Statistical predictive performance:
   - OOS R²
   - Pearson IC
   - Rank IC
   - year-by-year stability

2. Economic performance:
   - long top 20% / short bottom 20%
   - dollar-neutral, 100% gross exposure
   - turnover
   - 0, 5, and 10 bps transaction-cost assumptions
   - annualized return, volatility, Sharpe ratio, and maximum drawdown

No model specification or hyperparameter will be changed based on
test-period performance.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    ElasticNet,
)

from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

In [2]:
# Load the data
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(
    DATA_DIR / "alpha_dataset.csv",
    parse_dates=[
        "date",
        "next_market_date",
    ],
)

In [3]:
# Load linear parameters
with open(
    DATA_DIR / "linear_model_params.json",
    "r"
) as f:
    linear_params = json.load(f)

linear_params

{'ridge': {'alpha': 100000.0},
 'elastic_net': {'alpha': 3e-05, 'l1_ratio': 0.9}}

In [4]:
# Load XGBoost parameters
with open(
    DATA_DIR / "xgboost_params.json",
    "r"
) as f:
    xgb_params = json.load(f)

xgb_params

{'max_depth': 1,
 'learning_rate': 0.05,
 'n_estimators': 300,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 10,
 'reg_lambda': 10.0,
 'reg_alpha': 0.0}

## Define features, target and evaluation functions

In [5]:
MODEL_FEATURES = [
    "ret_1d_z",
    "mom_5_z",
    "mom_20_z",
    "mom_60_z",
    "vol_20_z",
    "vol_60_z",
    "price_ma20_gap_z",
    "dollar_volume_z20_z",
    "beta_60_z",
]

TARGET = "target_cs_1d"

In [6]:
required_cols = (
    MODEL_FEATURES
    + [
        TARGET,
        "fwd_ret_1d",
        "next_market_date",
    ]
)

missing_cols = (
    set(required_cols)
    - set(df.columns)
)

assert not missing_cols, (
    f"Missing columns: {missing_cols}"
)

In [7]:
def oos_r2(y_true, y_pred):
    """
    OOS R² relative to predicting zero cross-sectional return.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    model_sse = np.sum(
        (y_true - y_pred) ** 2
    )

    zero_sse = np.sum(
        y_true ** 2
    )

    return 1 - model_sse / zero_sse

In [8]:
def daily_ic(
    df,
    prediction_col,
    target_col=TARGET,
    method="spearman",
):
    """
    Calculate cross-sectional prediction/return correlation
    separately for each date.
    """

    dates = []
    ic_values = []

    for date, d in df.groupby("date"):

        valid = d[
            [prediction_col, target_col]
        ].dropna()

        if len(valid) < 5:
            continue

        if (
            valid[prediction_col].nunique() < 2
            or valid[target_col].nunique() < 2
        ):
            ic = np.nan

        else:
            ic = valid[prediction_col].corr(
                valid[target_col],
                method=method,
            )

        dates.append(date)
        ic_values.append(ic)

    return pd.Series(
        ic_values,
        index=pd.to_datetime(dates),
        name=f"{method}_ic",
    )

In [9]:
# General evaluation
def evaluate_predictions(
    data,
    predictions,
    model_name,
):
    """
    Evaluate prediction accuracy and cross-sectional ranking.
    """

    temp = data[
        ["date", "ticker", TARGET]
    ].copy()

    temp["prediction"] = predictions

    mse = mean_squared_error(
        temp[TARGET],
        temp["prediction"],
    )

    r2 = oos_r2(
        temp[TARGET],
        temp["prediction"],
    )

    pearson_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="pearson",
    )

    rank_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="spearman",
    )

    return {
        "model": model_name,
        "mse": mse,
        "oos_r2": r2,
        "mean_pearson_ic":
            pearson_ic.mean(),
        "mean_rank_ic":
            rank_ic.mean(),
        "rank_ic_std":
            rank_ic.std(),
        "pct_rank_ic_positive":
            (rank_ic > 0).mean(),
    }

## Function to creates the frozen models

In [10]:
def make_models(
    linear_params,
    xgb_params,
):
    """
    Create fresh model objects using the hyperparameters
    selected during the validation period.
    """

    models = {
        "ols": LinearRegression(),

        "ridge": Ridge(
            alpha=linear_params[
                "ridge"
            ]["alpha"]
        ),

        "elastic_net": ElasticNet(
            alpha=linear_params[
                "elastic_net"
            ]["alpha"],
            l1_ratio=linear_params[
                "elastic_net"
            ]["l1_ratio"],
            max_iter=100_000,
        ),

        "xgboost": XGBRegressor(
            objective="reg:squarederror",

            max_depth=xgb_params[
                "max_depth"
            ],

            learning_rate=xgb_params[
                "learning_rate"
            ],

            n_estimators=xgb_params[
                "n_estimators"
            ],

            subsample=xgb_params[
                "subsample"
            ],

            colsample_bytree=xgb_params[
                "colsample_bytree"
            ],

            min_child_weight=xgb_params[
                "min_child_weight"
            ],

            reg_lambda=xgb_params[
                "reg_lambda"
            ],

            reg_alpha=xgb_params[
                "reg_alpha"
            ],

            random_state=42,
            n_jobs=-1,
        ),
    }

    return models

## Annual expanding-window procedure

* For the 2022 predictions: Fit all eligible data before the first 2022 trading day and predict all 2022 dates

* For the 2023 predictions: Fit all eligible data through 2022 and predict all 2023 dates

* For the 2024 predictions: Fit all eligible data through 2023 and predict all 2024 dates

and so on.

Hyperparameters remain fixed throughout.

This mimics a researcher who periodically retrains the model as new realized returns become available, without using future observations.

## Generate walk-forward predictions

In [11]:
test_years = sorted(
    df.loc[
        df["split"] == "test",
        "date"
    ]
    .dt.year
    .unique()
)

test_years

[np.int32(2022),
 np.int32(2023),
 np.int32(2024),
 np.int32(2025),
 np.int32(2026)]

In [12]:
prediction_parts = []
refit_log = []

for year in test_years:

    # -------------------------------
    # 1. Data we need to predict
    # -------------------------------

    year_data = (
        df[
            (df["split"] == "test")
            & (df["date"].dt.year == year)
        ]
        .sort_values(
            ["date", "ticker"]
        )
        .copy()
    )

    first_prediction_date = (
        year_data["date"].min()
    )

    # -------------------------------
    # 2. Training data available
    #    before this prediction year
    # -------------------------------

    train_data = (
        df[
            df["next_market_date"]
            < first_prediction_date
        ]
        .dropna(
            subset=MODEL_FEATURES + [TARGET]
        )
        .sort_values(
            ["date", "ticker"]
        )
        .copy()
    )

    X_train = train_data[
        MODEL_FEATURES
    ]

    y_train = train_data[
        TARGET
    ]

    X_year = year_data[
        MODEL_FEATURES
    ]

    # -------------------------------
    # 3. Fresh frozen models
    # -------------------------------

    models = make_models(
        linear_params,
        xgb_params,
    )

    # Basic output for this year
    year_predictions = year_data[
        [
            "date",
            "next_market_date",
            "ticker",
            "fwd_ret_1d",
            TARGET,
        ]
    ].copy()

    # -------------------------------
    # 4. Fit and predict each model
    # -------------------------------

    for model_name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        year_predictions[
            f"{model_name}_pred"
        ] = model.predict(
            X_year
        )

    prediction_parts.append(
        year_predictions
    )

    # Record exactly what data each
    # annual model was trained on
    refit_log.append({
        "prediction_year":
            year,

        "first_prediction_date":
            first_prediction_date,

        "train_start":
            train_data["date"].min(),

        "train_end":
            train_data["date"].max(),

        "n_train_rows":
            len(train_data),

        "n_train_dates":
            train_data["date"].nunique(),

        "n_prediction_rows":
            len(year_data),
    })

In [13]:
# Combine
oos_predictions = (
    pd.concat(
        prediction_parts,
        ignore_index=True
    )
    .sort_values(
        ["date", "ticker"]
    )
    .reset_index(drop=True)
)

In [14]:
## Refit log
refit_log = pd.DataFrame(
    refit_log
)

refit_log

,prediction_year,first_prediction_date,train_start,train_end,n_train_rows,n_train_dates,n_prediction_rows
0,2022,2022-01-03,2010-03-31,2021-12-30,88800,2960,7530
1,2023,2023-01-03,2010-03-31,2022-12-29,96330,3211,7500
2,2024,2024-01-02,2010-03-31,2023-12-28,103830,3461,7560
3,2025,2025-01-02,2010-03-31,2024-12-30,111390,3713,7500
4,2026,2026-01-02,2010-03-31,2025-12-30,118890,3963,4860


## Inspect predictions

In [15]:
oos_predictions.head()

,date,next_market_date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
0,2022-01-03,2022-01-04,AAPL,-0.012692,-0.021930,0.000102,0.000023,0.000052,0.000339
1,2022-01-03,2022-01-04,AMGN,0.005073,-0.004165,0.000194,0.000025,0.000040,0.000161
2,2022-01-03,2022-01-04,BA,0.027759,0.018521,0.000010,0.000012,-0.000003,0.000098
3,2022-01-03,2022-01-04,BAC,0.039194,0.029956,-0.000339,-0.000131,-0.000175,-0.000373
4,2022-01-03,2022-01-04,C,0.007765,-0.001473,-0.000507,-0.000179,-0.000254,-0.000334


In [16]:
oos_predictions[
    [
        "ols_pred",
        "ridge_pred",
        "elastic_net_pred",
        "xgboost_pred",
    ]
].describe()

,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
count,3.495000e+04,3.495000e+04,3.495000e+04,34950.000000
mean,5.673262e-19,5.673441e-19,5.671735e-19,-0.000009
std,1.637073e-04,7.190085e-05,8.799391e-05,0.000389
min,-7.449749e-04,-3.628818e-04,-4.752187e-04,-0.005395
25%,-1.038833e-04,-4.526897e-05,-5.377233e-05,-0.000068
50%,-5.253300e-06,-1.765339e-07,4.290704e-06,0.000024
75%,9.702444e-05,4.391286e-05,5.697924e-05,0.000094
max,9.330825e-04,3.693347e-04,3.534571e-04,0.004542


In [17]:
print(
    "Dates:",
    oos_predictions["date"].min(),
    "to",
    oos_predictions["date"].max(),
)

print(
    "Number of dates:",
    oos_predictions["date"].nunique()
)

Dates: 2022-01-03 00:00:00 to 2026-08-25 00:00:00
Number of dates: 1165


## Overall test-period prediction performance

In [18]:
PREDICTION_COLS = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [19]:
test_results = []

for model_name, pred_col in (
    PREDICTION_COLS.items()
):

    metrics = evaluate_predictions(
        oos_predictions,
        oos_predictions[
            pred_col
        ].values,
        model_name,
    )

    test_results.append(
        metrics
    )

In [20]:
test_results = pd.DataFrame(
    test_results
)

test_results.sort_values(
    "mean_rank_ic",
    ascending=False
)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
2,Elastic Net,0.000261,-0.000076,-0.001756,-0.002113,0.245061,0.493562
1,Ridge,0.000261,-0.000053,-0.002751,-0.004002,0.267247,0.497854
3,XGBoost,0.000261,-0.000920,-0.006483,-0.004412,0.229230,0.499571
0,OLS,0.000261,-0.000222,-0.007267,-0.007998,0.251949,0.490129


## Performance by year

In [21]:
yearly_prediction_results = []

for year, d in (
    oos_predictions.groupby(
        oos_predictions["date"].dt.year
    )
):

    for model_name, pred_col in (
        PREDICTION_COLS.items()
    ):

        metrics = evaluate_predictions(
            d,
            d[pred_col].values,
            model_name,
        )

        metrics["year"] = year

        yearly_prediction_results.append(
            metrics
        )

In [22]:
yearly_prediction_results = pd.DataFrame(
    yearly_prediction_results
)

yearly_prediction_results[
    [
        "year",
        "model",
        "oos_r2",
        "mean_rank_ic",
        "pct_rank_ic_positive",
    ]
]

,year,model,oos_r2,mean_rank_ic,pct_rank_ic_positive
0,2022,OLS,-0.000332,-0.008589,0.505976
1,2022,Ridge,-0.000256,-0.020753,0.486056
2,2022,Elastic Net,-0.000283,-0.006737,0.509960
3,2022,XGBoost,-0.001032,-0.032045,0.450199
4,2023,OLS,-0.000566,-0.026778,0.460000
5,2023,Ridge,-0.000035,-0.015183,0.480000
6,2023,Elastic Net,-0.000051,-0.012579,0.468000
7,2023,XGBoost,-0.000094,-0.010692,0.488000
8,2024,OLS,-0.000253,-0.007359,0.480159
9,2024,Ridge,-0.000066,-0.000085,0.476190


In [23]:
yearly_rank_ic = (
    yearly_prediction_results
    .pivot(
        index="year",
        columns="model",
        values="mean_rank_ic",
    )
)

yearly_rank_ic

model,Elastic Net,OLS,Ridge,XGBoost
year,,,,
2022,-0.006737,-0.008589,-0.020753,-0.032045
2023,-0.012579,-0.026778,-0.015183,-0.010692
2024,0.002361,-0.007359,-0.000085,0.001388
2025,0.011441,0.008301,0.016827,0.024575
2026,-0.006671,-0.004246,0.000972,-0.005661


In [24]:
# Plot annual Rank IC
rank_ic_plot = (
    yearly_prediction_results[
        [
            "year",
            "model",
            "mean_rank_ic",
        ]
    ]
)

fig = px.line(
    rank_ic_plot,
    x="year",
    y="mean_rank_ic",
    color="model",
    markers=True,
    title="Out-of-Sample Rank IC by Year",
)

fig.add_hline(
    y=0,
    line_dash="dash",
)

fig.show()

---

## Convert scores into portfolio weights

Our portfolio rule is: $$ \text{Long} : \text{Top 20\%} $$
 $$ \text{Short} : \text{Bottom 20\% } $$

So for 30 stocks: $$ 30 \times 0.20 = 6 $$
So each day we'll typically hold:
* 6 long stocks
* 6 short stocks

We will use $ 100\% $ gross exposure: $$ \sum_{i} |w_i| = 1 $$
That means:  $$ \sum_{\text{long}} w_i = +0.5 $$
 and $$ \sum_{\text{short}} w_i = -0.5 $$
Thus the portfolio is: $$ \text{Dollar Neutral} $$
with $$ \sum_{i} w_i = 0 $$

## Portfolio construction function

In [25]:
def build_long_short_portfolio(
    predictions,
    prediction_col,
    long_fraction=0.20,
):
    """
    Construct a daily dollar-neutral long-short portfolio.

    Long:
        top long_fraction of predictions

    Short:
        bottom long_fraction of predictions

    Gross exposure:
        100%

    Long exposure:
        +50%

    Short exposure:
        -50%
    """

    records = []

    # Store yesterday's target weights so
    # we can approximate today's turnover.
    previous_weights = {}

    for date, d in (
        predictions
        .sort_values(
            ["date", "ticker"]
        )
        .groupby("date")
    ):

        d = d.dropna(
            subset=[
                prediction_col,
                "fwd_ret_1d",
            ]
        ).copy()

        n_stocks = len(d)

        if n_stocks < 2:
            continue

        n_side = max(
            1,
            int(
                np.floor(
                    n_stocks
                    * long_fraction
                )
            )
        )

        # Deterministic tie breaking using ticker.
        d = d.sort_values(
            [
                prediction_col,
                "ticker",
            ]
        )

        shorts = d.head(
            n_side
        )

        longs = d.tail(
            n_side
        )

        # All other stocks receive zero weight.
        current_weights = {
            ticker: 0.0
            for ticker in d["ticker"]
        }

        # Total long exposure = +0.5
        long_weight = (
            0.5 / n_side
        )

        # Total short exposure = -0.5
        short_weight = (
            -0.5 / n_side
        )

        for ticker in longs["ticker"]:
            current_weights[ticker] = (
                long_weight
            )

        for ticker in shorts["ticker"]:
            current_weights[ticker] = (
                short_weight
            )

        # Map weights back onto rows.
        d["weight"] = (
            d["ticker"]
            .map(current_weights)
        )

        # Portfolio return over the next
        # market trading interval.
        gross_return = (
            d["weight"]
            * d["fwd_ret_1d"]
        ).sum()

        # Separate contributions for diagnostics.
        long_return = (
            d.loc[
                d["weight"] > 0,
                "weight"
            ]
            * d.loc[
                d["weight"] > 0,
                "fwd_ret_1d"
            ]
        ).sum()

        short_return = (
            d.loc[
                d["weight"] < 0,
                "weight"
            ]
            * d.loc[
                d["weight"] < 0,
                "fwd_ret_1d"
            ]
        ).sum()

        # ----------------------------------
        # Approximate turnover
        #
        # Sum of dollars traded as a
        # fraction of portfolio capital.
        # ----------------------------------

        all_tickers = (
            set(previous_weights)
            | set(current_weights)
        )

        turnover = sum(
            abs(
                current_weights.get(
                    ticker,
                    0.0
                )
                -
                previous_weights.get(
                    ticker,
                    0.0
                )
            )
            for ticker in all_tickers
        )

        records.append({
            "signal_date":
                date,

            "return_date":
                d["next_market_date"].iloc[0],

            "gross_return":
                gross_return,

            "long_return":
                long_return,

            "short_return":
                short_return,

            "turnover":
                turnover,

            "n_long":
                n_side,

            "n_short":
                n_side,
        })

        previous_weights = (
            current_weights.copy()
        )

    return pd.DataFrame(
        records
    )

## Build all four portfolios

In [26]:
portfolios = {}

for model_name, pred_col in (
    PREDICTION_COLS.items()
):

    portfolios[model_name] = (
        build_long_short_portfolio(
            oos_predictions,
            prediction_col=pred_col,
            long_fraction=0.20,
        )
    )

In [27]:
# Inspect XGBoost
portfolios[
    "XGBoost"
].head()

,signal_date,return_date,gross_return,long_return,short_return,turnover,n_long,n_short
0,2022-01-03,2022-01-04,-0.017752,-0.006039,-0.011714,1.000000,6,6
1,2022-01-04,2022-01-05,0.003881,-0.000813,0.004695,0.666667,6,6
2,2022-01-05,2022-01-06,-0.000835,0.002112,-0.002947,0.666667,6,6
3,2022-01-06,2022-01-07,0.004290,-0.001710,0.006000,1.000000,6,6
4,2022-01-07,2022-01-10,0.001354,0.005334,-0.003980,0.833333,6,6


In [28]:
# Check exposure indirectly
portfolios[
    "XGBoost"
][
    [
        "gross_return",
        "turnover",
        "n_long",
        "n_short",
    ]
].describe()

,gross_return,turnover,n_long,n_short
count,1165.000000,1165.000000,1165.0,1165.0
mean,-0.000025,0.942632,6.0,6.0
std,0.005950,0.266628,0.0,0.0
min,-0.034746,0.166667,6.0,6.0
25%,-0.002966,0.833333,6.0,6.0
50%,0.000117,1.000000,6.0,6.0
75%,0.003251,1.166667,6.0,6.0
max,0.026969,1.666667,6.0,6.0


## Transaction costs

We'll test: $$ 0, 5, 10 \text{ bps} $$ per dollar traded

In [29]:
COST_BPS = [ 0, 5, 10 ]

for model_name, p in (
    portfolios.items()
):

    for cost_bps in COST_BPS:

        cost_rate = (
            cost_bps / 10_000
        )

        p[
            f"net_return_{cost_bps}bps"
        ] = (
            p["gross_return"]
            -
            cost_rate
            * p["turnover"]
        )

## Backtest Audit Checks

Before evaluating portfolio performance, verify that:

1. `fwd_ret_1d` corresponds exactly to the adjusted-close return from the
   signal date to the stated next market trading date.

2. Transaction costs are applied correctly as:

       net return = gross return - turnover × cost rate

These checks validate implementation only. They do not change the model,
portfolio construction, or test-period results.

## Audit 1 - verify forward return alignment

In [50]:
# Load the original OHLCV data so we can independently
# reconstruct the next-day return from prices.

raw_prices = pd.read_csv(
    DATA_DIR / "raw_prices.csv",
    parse_dates=["date"]
)

# We only need adjusted close for this check.
price_lookup = raw_prices[
    [
        "date",
        "ticker",
        "adj_close",
    ]
].copy()

In [51]:
# Start with the signal date, next trading date,
# ticker, and stored forward return.

return_audit = oos_predictions[
    [
        "date",
        "next_market_date",
        "ticker",
        "fwd_ret_1d",
    ]
].copy()


# --------------------------------------------------
# Attach the adjusted close on the SIGNAL date.
# --------------------------------------------------

current_prices = price_lookup.rename(
    columns={
        "adj_close": "adj_close_t"
    }
)

return_audit = return_audit.merge(
    current_prices,
    on=["date", "ticker"],
    how="left",
)


# --------------------------------------------------
# Attach the adjusted close on the NEXT market date.
# --------------------------------------------------

next_prices = price_lookup.rename(
    columns={
        "date": "next_market_date",
        "adj_close": "adj_close_next",
    }
)

return_audit = return_audit.merge(
    next_prices,
    on=[
        "next_market_date",
        "ticker",
    ],
    how="left",
)


# --------------------------------------------------
# Independently reconstruct the forward return.
# --------------------------------------------------

return_audit[
    "reconstructed_fwd_ret"
] = (
    return_audit["adj_close_next"]
    / return_audit["adj_close_t"]
    - 1
)


# Difference between our independently calculated
# return and the stored return used in the backtest.
return_audit[
    "difference"
] = (
    return_audit["reconstructed_fwd_ret"]
    - return_audit["fwd_ret_1d"]
)

In [52]:
return_audit[
    [
        "date",
        "next_market_date",
        "ticker",
        "adj_close_t",
        "adj_close_next",
        "fwd_ret_1d",
        "reconstructed_fwd_ret",
        "difference",
    ]
].head(10)

,date,next_market_date,ticker,adj_close_t,adj_close_next,fwd_ret_1d,reconstructed_fwd_ret,difference
0,2022-01-03,2022-01-04,AAPL,177.786392,175.529999,-0.012692,-0.012692,-2.081668e-17
1,2022-01-03,2022-01-04,AMGN,195.656067,196.648621,0.005073,0.005073,7.025630e-17
2,2022-01-03,2022-01-04,BA,207.860001,213.630005,0.027759,0.027759,-1.734723e-16
3,2022-01-03,2022-01-04,BAC,41.265347,42.882717,0.039194,0.039194,6.938894e-18
4,2022-01-03,2022-01-04,C,53.583778,53.999870,0.007765,0.007765,1.821460e-17
5,2022-01-03,2022-01-04,CAT,190.991135,201.214294,0.053527,0.053527,-1.318390e-16
6,2022-01-03,2022-01-04,CSCO,55.290710,53.934647,-0.024526,-0.024526,-6.938894e-18
7,2022-01-03,2022-01-04,CVX,98.725578,100.521935,0.018195,0.018195,-1.561251e-16
8,2022-01-03,2022-01-04,GS,353.786469,364.659668,0.030734,0.030734,8.673617e-17
9,2022-01-03,2022-01-04,HD,364.155334,367.898071,0.010278,0.010278,7.632783e-17


In [53]:
print(
    "Maximum absolute difference:",
    return_audit["difference"]
    .abs()
    .max()
)

print(
    "Missing current prices:",
    return_audit["adj_close_t"]
    .isna()
    .sum()
)

print(
    "Missing next prices:",
    return_audit["adj_close_next"]
    .isna()
    .sum()
)

Maximum absolute difference: 5.425347671117464e-16
Missing current prices: 0
Missing next prices: 0


In [54]:
assert return_audit[
    "adj_close_t"
].notna().all()

assert return_audit[
    "adj_close_next"
].notna().all()

assert np.allclose(
    return_audit[
        "fwd_ret_1d"
    ],
    return_audit[
        "reconstructed_fwd_ret"
    ],
    atol=1e-12,
)

print(
    "PASS: All forward returns are correctly "
    "aligned with the next market trading date."
)

PASS: All forward returns are correctly aligned with the next market trading date.


## Audit Cell 2 - Inspect one forward return manually

In [55]:
check_ticker = "AAPL"
check_date = pd.Timestamp(
    "2025-06-02"
)

check_row = return_audit[
    (return_audit["ticker"] == check_ticker)
    & (return_audit["date"] == check_date)
].iloc[0]

print(
    "Ticker:",
    check_row["ticker"]
)

print(
    "Signal date:",
    check_row["date"]
)

print(
    "Next market date:",
    check_row["next_market_date"]
)

print(
    "Adjusted close at t:",
    check_row["adj_close_t"]
)

print(
    "Adjusted close at t+1:",
    check_row["adj_close_next"]
)

print(
    "Stored forward return:",
    check_row["fwd_ret_1d"]
)

print(
    "Reconstructed forward return:",
    check_row["reconstructed_fwd_ret"]
)

Ticker: AAPL
Signal date: 2025-06-02 00:00:00
Next market date: 2025-06-03 00:00:00
Adjusted close at t: 200.72984313964844
Adjusted close at t+1: 202.29232788085935
Stored forward return: 0.0077840181448451
Reconstructed forward return: 0.007784018144844973


## Audit cell 3 - verify transaction-cost calculation

In [56]:
# Select one portfolio for an independent
# transaction-cost calculation.

cost_audit = (
    portfolios["XGBoost"]
    .copy()
)

COST_RATE = 5 / 10_000

In [57]:
cost_audit[
    "expected_cost_5bps"
] = (
    cost_audit["turnover"]
    * COST_RATE
)

cost_audit[
    "expected_net_return_5bps"
] = (
    cost_audit["gross_return"]
    - cost_audit[
        "expected_cost_5bps"
    ]
)

cost_audit[
    "difference"
] = (
    cost_audit[
        "expected_net_return_5bps"
    ]
    - cost_audit[
        "net_return_5bps"
    ]
)

In [58]:
cost_audit[
    [
        "signal_date",
        "gross_return",
        "turnover",
        "expected_cost_5bps",
        "net_return_5bps",
        "expected_net_return_5bps",
        "difference",
    ]
].head(10)

,signal_date,gross_return,turnover,expected_cost_5bps,net_return_5bps,expected_net_return_5bps,difference
0,2022-01-03,-0.017752,1.000000,0.000500,-0.018252,-0.018252,0.0
1,2022-01-04,0.003881,0.666667,0.000333,0.003548,0.003548,0.0
2,2022-01-05,-0.000835,0.666667,0.000333,-0.001168,-0.001168,0.0
3,2022-01-06,0.004290,1.000000,0.000500,0.003790,0.003790,0.0
4,2022-01-07,0.001354,0.833333,0.000417,0.000938,0.000938,0.0
5,2022-01-10,-0.002374,1.166667,0.000583,-0.002957,-0.002957,0.0
6,2022-01-11,-0.002375,1.166667,0.000583,-0.002958,-0.002958,0.0
7,2022-01-12,0.003363,1.333333,0.000667,0.002696,0.002696,0.0
8,2022-01-13,-0.001293,1.166667,0.000583,-0.001877,-0.001877,0.0
9,2022-01-14,-0.002397,1.166667,0.000583,-0.002981,-0.002981,0.0


In [59]:
print(
    "Maximum transaction-cost difference:",
    cost_audit["difference"]
    .abs()
    .max()
)

Maximum transaction-cost difference: 0.0


In [60]:
assert np.allclose(
    cost_audit[
        "net_return_5bps"
    ],
    cost_audit[
        "expected_net_return_5bps"
    ],
    atol=1e-12,
)

print(
    "PASS: 5 bps transaction costs "
    "are being applied correctly."
)

PASS: 5 bps transaction costs are being applied correctly.


## Audit cell 4 - verify all models and all cost assumptions

In [61]:
for model_name, p in portfolios.items():

    for cost_bps in COST_BPS:

        cost_rate = (
            cost_bps / 10_000
        )

        expected_net = (
            p["gross_return"]
            - p["turnover"]
            * cost_rate
        )

        actual_net = (
            p[
                f"net_return_{cost_bps}bps"
            ]
        )

        assert np.allclose(
            actual_net,
            expected_net,
            atol=1e-12,
        )

        print(
            f"PASS: {model_name}, "
            f"{cost_bps} bps"
        )

PASS: OLS, 0 bps
PASS: OLS, 5 bps
PASS: OLS, 10 bps
PASS: Ridge, 0 bps
PASS: Ridge, 5 bps
PASS: Ridge, 10 bps
PASS: Elastic Net, 0 bps
PASS: Elastic Net, 5 bps
PASS: Elastic Net, 10 bps
PASS: XGBoost, 0 bps
PASS: XGBoost, 5 bps
PASS: XGBoost, 10 bps


## Audit 5 - verify turnover arithmetic on one date

In [62]:
# Choose two consecutive signal dates.

audit_dates = (
    oos_predictions["date"]
    .drop_duplicates()
    .sort_values()
    .iloc[100:102]
    .tolist()
)

date_1 = audit_dates[0]
date_2 = audit_dates[1]

print(date_1)
print(date_2)

2022-05-26 00:00:00
2022-05-27 00:00:00


In [63]:
def get_target_weights(
    predictions,
    date,
    prediction_col,
    long_fraction=0.20,
):

    d = predictions[
        predictions["date"] == date
    ].copy()

    d = d.dropna(
        subset=[
            prediction_col,
            "fwd_ret_1d",
        ]
    )

    d = d.sort_values(
        [
            prediction_col,
            "ticker",
        ]
    )

    n_stocks = len(d)

    n_side = max(
        1,
        int(
            np.floor(
                n_stocks
                * long_fraction
            )
        )
    )

    weights = pd.Series(
        0.0,
        index=d["ticker"]
    )

    short_tickers = (
        d.head(n_side)["ticker"]
    )

    long_tickers = (
        d.tail(n_side)["ticker"]
    )

    weights.loc[
        short_tickers
    ] = -0.5 / n_side

    weights.loc[
        long_tickers
    ] = 0.5 / n_side

    return weights

In [64]:
w1 = get_target_weights(
    oos_predictions,
    date_1,
    "xgboost_pred",
)

w2 = get_target_weights(
    oos_predictions,
    date_2,
    "xgboost_pred",
)

In [65]:
weight_check = pd.concat(
    [
        w1.rename("weight_day_1"),
        w2.rename("weight_day_2"),
    ],
    axis=1,
).fillna(0)

In [66]:
weight_check[
    "weight_change"
] = (
    weight_check[
        "weight_day_2"
    ]
    - weight_check[
        "weight_day_1"
    ]
)

weight_check[
    "abs_weight_change"
] = (
    weight_check[
        "weight_change"
    ].abs()
)

weight_check.sort_values(
    "abs_weight_change",
    ascending=False
)

,weight_day_1,weight_day_2,weight_change,abs_weight_change
ticker,,,,
MS,-0.083333,0.000000,0.083333,0.083333
BAC,-0.083333,0.000000,0.083333,0.083333
WFC,-0.083333,0.000000,0.083333,0.083333
MRK,0.000000,0.083333,0.083333,0.083333
C,0.000000,0.083333,0.083333,0.083333
TXN,0.000000,-0.083333,-0.083333,0.083333
AMGN,0.083333,0.000000,-0.083333,0.083333
UNH,0.000000,-0.083333,-0.083333,0.083333
PEP,0.000000,-0.083333,-0.083333,0.083333


In [67]:
manual_turnover = (
    weight_check[
        "abs_weight_change"
    ].sum()
)

print(
    "Manual turnover:",
    manual_turnover
)

Manual turnover: 0.8333333333333334


In [68]:
stored_turnover = (
    portfolios["XGBoost"]
    .loc[
        portfolios["XGBoost"][
            "signal_date"
        ] == date_2,
        "turnover"
    ]
    .iloc[0]
)

print(
    "Stored turnover:",
    stored_turnover
)

print(
    "Difference:",
    manual_turnover
    - stored_turnover
)

Stored turnover: 0.8333333333333333
Difference: 1.1102230246251565e-16


In [69]:
assert np.isclose(
    manual_turnover,
    stored_turnover,
    atol=1e-12,
)

print(
    "PASS: Portfolio turnover "
    "is calculated correctly."
)

PASS: Portfolio turnover is calculated correctly.


### Audit conclusion

All implementation checks passed.

- Stored forward returns exactly match independently reconstructed
  adjusted-close returns from the signal date to the next market trading day.
- Portfolio weights are dollar-neutral with 100% gross exposure.
- Portfolio return aggregation is consistent with the underlying stock returns.
- Turnover calculations match independently reconstructed changes in target weights.
- Transaction costs are correctly applied as turnover multiplied by the assumed
  cost per dollar traded.

Therefore, the poor out-of-sample results are not attributable to an obvious
return-alignment, portfolio-weighting, turnover, or transaction-cost implementation error.

The backtest remains subject to the simplifying execution assumptions described below.

## Performance-statistics function

In [30]:
def max_drawdown(returns):
    """
    Maximum peak-to-trough drawdown of
    compounded portfolio wealth.
    """

    wealth = (
        1 + returns
    ).cumprod()

    running_peak = (
        wealth.cummax()
    )

    drawdown = (
        wealth / running_peak
        - 1
    )

    return drawdown.min()

In [31]:
def performance_stats(
    returns,
    turnover=None,
):
    """
    Calculate annualized portfolio statistics.

    Assumes approximately 252 trading days/year.
    """

    returns = (
        pd.Series(returns)
        .dropna()
    )

    n = len(returns)

    if n == 0:
        return {}

    # Geometric annualized return
    annual_return = (
        (1 + returns).prod()
        ** (252 / n)
        - 1
    )

    annual_vol = (
        returns.std()
        * np.sqrt(252)
    )

    if returns.std() > 0:

        sharpe = (
            returns.mean()
            / returns.std()
            * np.sqrt(252)
        )

    else:
        sharpe = np.nan

    result = {
        "annual_return":
            annual_return,

        "annual_vol":
            annual_vol,

        "sharpe":
            sharpe,

        "max_drawdown":
            max_drawdown(
                returns
            ),
    }

    if turnover is not None:

        result[
            "avg_daily_turnover"
        ] = np.mean(
            turnover
        )

        result[
            "annualized_turnover"
        ] = (
            np.mean(turnover)
            * 252
        )

    return result

## Final portfolio-results table

In [32]:
portfolio_results = []

for model_name, p in (
    portfolios.items()
):

    for cost_bps in COST_BPS:

        return_col = (
            f"net_return_{cost_bps}bps"
        )

        stats = performance_stats(
            p[return_col],
            turnover=p["turnover"],
        )

        stats["model"] = (
            model_name
        )

        stats["cost_bps"] = (
            cost_bps
        )

        portfolio_results.append(
            stats
        )

In [33]:
portfolio_results = pd.DataFrame(
    portfolio_results
)

portfolio_results = (
    portfolio_results[
        [
            "model",
            "cost_bps",
            "annual_return",
            "annual_vol",
            "sharpe",
            "max_drawdown",
            "avg_daily_turnover",
            "annualized_turnover",
        ]
    ]
)

portfolio_results

,model,cost_bps,annual_return,annual_vol,sharpe,max_drawdown,avg_daily_turnover,annualized_turnover
0,OLS,0,-0.056809,0.100289,-0.533051,-0.241871,0.920029,231.847210
1,OLS,5,-0.160106,0.100423,-1.686695,-0.552125,0.920029,231.847210
2,OLS,10,-0.252133,0.100600,-2.836044,-0.737344,0.920029,231.847210
3,Ridge,0,-0.019750,0.111170,-0.123873,-0.169196,0.982117,247.493562
4,Ridge,5,-0.133890,0.111224,-1.236399,-0.485909,0.982117,247.493562
5,Ridge,10,-0.234790,0.111321,-2.346937,-0.709045,0.982117,247.493562
6,Elastic Net,0,-0.033441,0.098601,-0.295627,-0.192787,1.116166,281.273820
7,Elastic Net,5,-0.160296,0.098552,-1.722815,-0.553985,1.116166,281.273820
8,Elastic Net,10,-0.270563,0.098552,-3.149846,-0.766941,1.116166,281.273820
9,XGBoost,0,-0.010593,0.094448,-0.065521,-0.186702,0.942632,237.543348


In [48]:
check_date = pd.Timestamp("2025-06-02")

d = (
    oos_predictions[
        oos_predictions["date"] == check_date
    ]
    .sort_values("xgboost_pred")
    .copy()
)

n_side = 6

d["weight"] = 0.0

d.loc[
    d.index[:n_side],
    "weight"
] = -0.5 / n_side

d.loc[
    d.index[-n_side:],
    "weight"
] = 0.5 / n_side

d[
    [
        "ticker",
        "xgboost_pred",
        "fwd_ret_1d",
        "weight",
    ]
]

,ticker,xgboost_pred,fwd_ret_1d,weight
25657,CVX,-0.000405,0.012406,-0.083333
25679,XOM,-0.000346,0.007278,-0.083333
25673,PG,-0.000294,-0.005543,-0.083333
25651,AMGN,-0.000105,0.003813,-0.083333
25655,CAT,-0.000092,0.013723,-0.083333
25670,ORCL,-0.000088,0.015429,-0.083333
25664,KO,-0.000048,-0.011667,0.000000
25665,MCD,-0.000037,-0.000927,0.000000
25671,PEP,-0.000034,0.007181,0.000000
25678,WMT,-0.000031,0.002105,0.000000


In [49]:
print(
    "Net exposure:",
    d["weight"].sum()
)

print(
    "Gross exposure:",
    d["weight"].abs().sum()
)

print(
    "Portfolio return:",
    (
        d["weight"]
        * d["fwd_ret_1d"]
    ).sum()
)

Net exposure: -8.326672684688674e-17
Gross exposure: 1.0000000000000002
Portfolio return: 0.0003875892853026249


In [ ]:
expected_net = (
    gross_return
    - turnover * 5 / 10_000
)

## Cost sensitivity table

It answers the question: Does the strategy survive plausible trading costs?

In [34]:
sharpe_by_cost = (
    portfolio_results
    .pivot(
        index="model",
        columns="cost_bps",
        values="sharpe",
    )
)

sharpe_by_cost.columns = [
    f"{c} bps"
    for c in sharpe_by_cost.columns
]

sharpe_by_cost

,0 bps,5 bps,10 bps
model,,,
Elastic Net,-0.295627,-1.722815,-3.149846
OLS,-0.533051,-1.686695,-2.836044
Ridge,-0.123873,-1.236399,-2.346937
XGBoost,-0.065521,-1.321907,-2.574829


## Cumulative returns at 5 bps

In [35]:
portfolio_plot_parts = []

for model_name, p in (
    portfolios.items()
):

    temp = p[
        [
            "return_date",
            "net_return_5bps",
        ]
    ].copy()

    temp["wealth"] = (
        1
        + temp[
            "net_return_5bps"
        ]
    ).cumprod()

    temp["model"] = (
        model_name
    )

    portfolio_plot_parts.append(
        temp
    )

In [36]:
# Combine
portfolio_plot = pd.concat(
    portfolio_plot_parts,
    ignore_index=True
)

In [37]:
# Plot
fig = px.line(
    portfolio_plot,
    x="return_date",
    y="wealth",
    color="model",
    title=(
        "Walk-Forward Long-Short Portfolio "
        "Growth — 5 bps Transaction Costs"
    ),
)

fig.show()

## Portfolio performance by year

In [38]:
yearly_portfolio_results = []

for model_name, p in (
    portfolios.items()
):

    p = p.copy()

    p["year"] = (
        pd.to_datetime(
            p["return_date"]
        ).dt.year
    )

    for year, d in p.groupby(
        "year"
    ):

        stats = performance_stats(
            d["net_return_5bps"],
            turnover=d["turnover"],
        )

        stats["model"] = (
            model_name
        )

        stats["year"] = (
            year
        )

        yearly_portfolio_results.append(
            stats
        )

In [39]:
# Create table
yearly_portfolio_results = (
    pd.DataFrame(
        yearly_portfolio_results
    )
)

yearly_portfolio_results[
    [
        "year",
        "model",
        "annual_return",
        "annual_vol",
        "sharpe",
        "max_drawdown",
    ]
]

,year,model,annual_return,annual_vol,sharpe,max_drawdown
0,2022,OLS,-0.187873,0.107181,-1.887340,-0.176450
1,2023,OLS,-0.124948,0.068777,-1.905909,-0.160965
2,2024,OLS,-0.140005,0.081218,-1.816070,-0.155238
3,2025,OLS,-0.108664,0.104646,-1.047154,-0.115308
4,2026,OLS,-0.269153,0.142721,-2.124539,-0.215825
5,2022,Ridge,-0.225457,0.111477,-2.235074,-0.222656
6,2023,Ridge,-0.106219,0.070208,-1.564153,-0.174728
7,2024,Ridge,-0.121420,0.090531,-1.384379,-0.138759
8,2025,Ridge,-0.008131,0.112285,-0.017048,-0.070230
9,2026,Ridge,-0.221701,0.172914,-1.362521,-0.196882


In [40]:
# Pivot Sharpe
yearly_sharpe = (
    yearly_portfolio_results
    .pivot(
        index="year",
        columns="model",
        values="sharpe",
    )
)

yearly_sharpe

model,Elastic Net,OLS,Ridge,XGBoost
year,,,,
2022,-2.361262,-1.887340,-2.235074,-3.051553
2023,-1.589169,-1.905909,-1.564153,-1.606651
2024,-1.567042,-1.816070,-1.384379,-0.486991
2025,-0.852017,-1.047154,-0.017048,0.005280
2026,-2.353248,-2.124539,-1.362521,-1.730028


## Long vs short contributions

In [41]:
leg_results = []

for model_name, p in (
    portfolios.items()
):

    leg_results.append({
        "model":
            model_name,

        "annualized_long_contribution":
            p["long_return"].mean()
            * 252,

        "annualized_short_contribution":
            p["short_return"].mean()
            * 252,

        "annualized_gross_return":
            p["gross_return"].mean()
            * 252,
    })

In [42]:
# Table
leg_results = pd.DataFrame(
    leg_results
)

leg_results

,model,annualized_long_contribution,annualized_short_contribution,annualized_gross_return
0,OLS,0.023481,-0.076940,-0.053459
1,Ridge,0.036258,-0.050028,-0.013771
2,Elastic Net,0.017969,-0.047118,-0.029149
3,XGBoost,0.063177,-0.069365,-0.006188


## Diagnostic 1 - Validation vs test Rank IC

It shows the signal decay, i.e., it visually shows that the signal did not persist out of sample

In [ ]:
# Load the validation predictions saved form notebook 04

validation_predictions = pd.read_csv(
    DATA_DIR / "model_validation_predictions.csv",
    parse_dates=["date"]
)

In [71]:
# Use the same prediction-column mapping

PREDICTION_COLS = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [72]:
# Compute validation and test Rank IC

ic_period_comparison = []

for model_name, pred_col in PREDICTION_COLS.items():

    # Validation-period IC
    val_ic = daily_ic(
        validation_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    # Test-period IC
    test_ic = daily_ic(
        oos_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    ic_period_comparison.append({
        "model": model_name,
        "validation_rank_ic": val_ic.mean(),
        "test_rank_ic": test_ic.mean(),
    })


ic_period_comparison = pd.DataFrame(
    ic_period_comparison
)

ic_period_comparison

,model,validation_rank_ic,test_rank_ic
0,OLS,0.003027,-0.007998
1,Ridge,0.009774,-0.004002
2,Elastic Net,0.008646,-0.002113
3,XGBoost,0.016245,-0.004412


In [73]:
# Plot 
ic_long = ic_period_comparison.melt(
    id_vars="model",
    var_name="period",
    value_name="rank_ic",
)

fig = px.bar(
    ic_long,
    x="model",
    y="rank_ic",
    color="period",
    barmode="group",
    title="Validation vs Test Rank IC",
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

## Diagnostic 2 - Rolling 60-day test Rank IC

This shows how predictability changes through time.

In [74]:
test_ic_frames = []

for model_name, pred_col in PREDICTION_COLS.items():

    ic = daily_ic(
        oos_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    temp = pd.DataFrame({
        "date": ic.index,
        "rank_ic": ic.values,
        "model": model_name,
    })

    test_ic_frames.append(temp)


test_ic_series = pd.concat(
    test_ic_frames,
    ignore_index=True
)

In [75]:
# Sort

test_ic_series = (
    test_ic_series
    .sort_values(
        ["model", "date"]
    )
)

In [76]:
# Calculate the rolling mean

test_ic_series["rank_ic_60d"] = (
    test_ic_series
    .groupby("model")["rank_ic"]
    .transform(
        lambda s:
        s.rolling(
            window=60,
            min_periods=20
        ).mean()
    )
)

In [77]:
# Plot 

fig = px.line(
    test_ic_series,
    x="date",
    y="rank_ic_60d",
    color="model",
    title="60-Day Rolling Rank IC — Test Period",
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

## Diagnostic 3 - Turnover

In [78]:
# Create a combined dataframe

turnover_data = []

for model_name, p in portfolios.items():

    temp = p[
        [
            "signal_date",
            "turnover",
        ]
    ].copy()

    temp["model"] = model_name

    turnover_data.append(temp)


turnover_data = pd.concat(
    turnover_data,
    ignore_index=True
)

In [79]:
# Summary table

turnover_summary = (
    turnover_data
    .groupby("model")["turnover"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
)

turnover_summary

,mean,median,std,min,max
model,,,,,
Elastic Net,1.116166,1.166667,0.277980,0.000000,2.000000
OLS,0.920029,0.833333,0.264282,0.166667,1.833333
Ridge,0.982117,1.000000,0.273634,0.166667,1.833333
XGBoost,0.942632,1.000000,0.266628,0.166667,1.666667


In [80]:
# Add annualized turnover

turnover_summary[
    "annualized_turnover"
] = (
    turnover_summary["mean"] * 252
)

turnover_summary

,mean,median,std,min,max,annualized_turnover
model,,,,,,
Elastic Net,1.116166,1.166667,0.277980,0.000000,2.000000,281.273820
OLS,0.920029,0.833333,0.264282,0.166667,1.833333,231.847210
Ridge,0.982117,1.000000,0.273634,0.166667,1.833333,247.493562
XGBoost,0.942632,1.000000,0.266628,0.166667,1.666667,237.543348


In [81]:
# Plot the distribution

fig = px.box(
    turnover_data,
    x="model",
    y="turnover",
    title="Daily Portfolio Turnover — Test Period",
)

fig.show()

## Save final predictions

In [43]:
oos_predictions.to_csv(
    DATA_DIR
    / "oos_predictions.csv",
    index=False,
)

## Save portfolio daily returns

In [44]:
portfolio_output = []

for model_name, p in (
    portfolios.items()
):

    temp = p.copy()

    temp["model"] = (
        model_name
    )

    portfolio_output.append(
        temp
    )

In [45]:
portfolio_output = pd.concat(
    portfolio_output,
    ignore_index=True
)

portfolio_output.to_csv(
    DATA_DIR
    / "portfolio_returns.csv",
    index=False,
)

In [47]:
# Save summary
portfolio_results.to_csv(
    DATA_DIR
    / "portfolio_summary.csv",
    index=False,
)

## Execution assumption

Features at date t use end-of-day information through date t, while
`fwd_ret_1d` measures the adjusted-close return from t to the next trading day.

The research backtest therefore assumes that the signal can be implemented
at or near the date-t close after the feature values are observed.

This is a simplifying execution assumption rather than a detailed simulation
of intraday order placement. A production implementation would need to
specify an executable signal cutoff and execution price explicitly, for
example by using pre-close information, lagging signals, or modeling
next-open execution.

Transaction costs are also simplified as a fixed number of basis points per
dollar traded, and turnover is calculated from target portfolio weights
rather than drift-adjusted pre-trade weights.